# Module 1 — Data Pipeline: scrape → clean → convert → store → query

**Goal:** turn raw product listings from [books.toscrape.com](https://books.toscrape.com) (a public site built for scraping practice) into a clean, normalized SQLite database, then query it with SQL **and** with pandas.

| Step | What happens | Output |
|---|---|---|
| 1. Scrape | `requests` downloads each category page, `BeautifulSoup` pulls out the fields | `data/raw_books.csv` |
| 2. Clean | text → proper types (`price_gbp` float, `rating` int, `in_stock` bool) | cleaned DataFrame |
| 3. Convert | `price_inr = price_gbp × 105.50` (fixed project rate) | `price_inr` column |
| 4. Store | two-table schema `categories` ⟷ `books` (PK/FK) | `data/books.db` |
| 5. Query | 6 SQL queries + `pd.read_sql` vs `pd.merge` check | `outputs/query_results.md` |

Run this notebook top to bottom (**Kernel → Restart & Run All**). It needs internet access to books.toscrape.com.

## 0. Setup

In [1]:
import os
import re
import sqlite3
import time
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Where to scrape from. The env-var override exists only so the code can be tested
# against a local copy of the site; normally it is the real public site.
BASE_URL = os.environ.get("BOOKS_BASE_URL", "https://books.toscrape.com/")

# Scope: every book in these categories (all their pages).  Together they hold 125 books,
# comfortably above the 60-book / 3-category minimum.
CATEGORIES = ["Mystery", "Historical Fiction", "Fantasy", "Poetry"]

# Project-defined fixed conversion rate (NOT a live market rate). 1 GBP = 105.50 INR
GBP_TO_INR = 105.50

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / "books.db"

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 45)
print("Scraping from:", BASE_URL)

Scraping from: https://books.toscrape.com/


## 1. Scrape

Three small functions:

* `fetch(url)` — downloads one page. It sends a User-Agent, uses a timeout, **checks the HTTP status code** (`raise_for_status()`), retries up to 3 times, and pauses briefly between requests so we are polite to the server.
* `get_category_urls()` — reads the category list in the left sidebar of the home page, so we never hard-code category URLs.
* `scrape_category()` — reads every book card on a category page, then follows the **next** button until there are no more pages.

On each book card (`<article class="product_pod">`) the fields live here:

| Field | HTML location |
|---|---|
| title | `h3 > a` → `title` attribute (the visible text is truncated with “...”) |
| price | `p.price_color` e.g. `£51.77` |
| star_rating | 2nd CSS class of `p.star-rating` e.g. `star-rating Three` → `"Three"` |
| availability | text of `p.instock.availability` e.g. `In stock` |

In [2]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (capstone scraping exercise)"})


def fetch(url, retries=3, pause=0.3):
    # Download one page and return it as a BeautifulSoup object.
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15)
            resp.raise_for_status()          # turns 4xx/5xx status codes into an exception
            resp.encoding = "utf-8"          # the site is UTF-8; without this '£' can become 'Â£'
            time.sleep(pause)                # be polite: don't hammer the server
            return BeautifulSoup(resp.text, "html.parser")
        except requests.RequestException as err:
            print(f"  attempt {attempt}/{retries} failed for {url}: {err}")
            time.sleep(1.5 * attempt)
    raise RuntimeError(f"Could not download {url} after {retries} attempts")


def get_category_urls():
    # Return {category name: absolute URL} from the home page sidebar.
    soup = fetch(BASE_URL)
    links = soup.select("div.side_categories ul li ul li a")
    return {a.get_text(strip=True): urljoin(BASE_URL, a["href"]) for a in links}


def parse_card(card, category):
    # Extract the raw text fields from one <article class="product_pod">.
    link = card.select_one("h3 a")
    rating_tag = card.select_one("p.star-rating")
    rating_classes = [c for c in rating_tag.get("class", []) if c != "star-rating"] if rating_tag else []
    price_tag = card.select_one("p.price_color")
    avail_tag = card.select_one("p.instock.availability")
    return {
        "title": link.get("title") or link.get_text(strip=True),
        "price": price_tag.get_text(strip=True) if price_tag else None,
        "star_rating": rating_classes[0] if rating_classes else None,
        "availability": avail_tag.get_text(" ", strip=True) if avail_tag else None,
        "category": category,
    }


def scrape_category(name, url):
    rows, page = [], 1
    while url:
        soup = fetch(url)
        cards = soup.select("article.product_pod")
        rows.extend(parse_card(card, name) for card in cards)
        print(f"  {name}: page {page} -> {len(cards)} books")
        next_link = soup.select_one("li.next a")
        url = urljoin(url, next_link["href"]) if next_link else None   # relative -> absolute
        page += 1
    return rows

In [3]:
category_urls = get_category_urls()
print(f"Found {len(category_urls)} categories on the site")

missing = [c for c in CATEGORIES if c not in category_urls]
assert not missing, f"Categories not found on site: {missing}"

raw_rows = []
for cat in CATEGORIES:
    raw_rows.extend(scrape_category(cat, category_urls[cat]))

raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv(DATA_DIR / "raw_books.csv", index=False)   # keep an untouched copy of the raw scrape

print("\nRaw rows scraped:", len(raw_df))
print(raw_df["category"].value_counts().to_string())
assert len(raw_df) >= 60 and raw_df["category"].nunique() >= 3, "Scope requirement not met"
raw_df.head()

Found 50 categories on the site


  Mystery: page 1 -> 20 books


  Mystery: page 2 -> 12 books


  Historical Fiction: page 1 -> 20 books


  Historical Fiction: page 2 -> 6 books


  Fantasy: page 1 -> 20 books


  Fantasy: page 2 -> 20 books


  Fantasy: page 3 -> 8 books


  Poetry: page 1 -> 19 books

Raw rows scraped: 125
category
Fantasy               48
Mystery               32
Historical Fiction    26
Poetry                19


,title,price,star_rating,availability,category
0,Sharp Objects,£47.82,Four,In stock,Mystery
1,"In a Dark, Dark Wood",£19.63,One,In stock,Mystery
2,The Past Never Ends,£56.50,Four,In stock,Mystery
3,A Murder in Time,£16.64,One,In stock,Mystery
4,The Murder of Roger Ackroyd (Hercule Poir...,£44.10,Four,In stock,Mystery


## 2. Clean the scraped fields into proper types

Each parser returns a **missing value instead of crashing** when the text is unexpected:

| Raw column | Clean column | Rule |
|---|---|---|
| `price` `"£51.77"` | `price_gbp` (float) | regex pulls out the number; anything else → `NaN` |
| `star_rating` `"Three"` | `rating` (int 1–5) | lookup table One…Five → 1…5; anything else → `NaN` |
| `availability` `"In stock"` | `in_stock` (bool) | `"in stock"` → `True`, `"out of stock"` → `False`, anything else → missing |

**How unparseable rows are handled (my decision):**

* **Numeric fields (`price_gbp`, `rating`) → median imputation.** A bad price or rating on one card should not throw away an otherwise good product row. I use the **median of the same category** (falls back to the overall median if the whole category is missing). The median is used rather than the mean because it is not pulled around by a few very cheap or very expensive books. Imputed ratings are rounded to a whole star so `rating` stays an integer 1–5. An `imputed` flag records which rows were filled, so they stay traceable.
* **`in_stock` → drop the row.** It is a yes/no fact about a specific product; there is no sensible “median” of a boolean, and guessing would make the stock figures wrong.
* **Missing title or category → drop the row**, because the row cannot be identified or linked to the `categories` table.
* **Duplicates** (same title in the same category) are dropped so each product appears once.

In [4]:
RATING_MAP = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}


def parse_price(text):
    # '£51.77' -> 51.77 ; anything unparseable -> NaN
    if not isinstance(text, str):
        return np.nan
    match = re.search(r"\d+(?:\.\d+)?", text.replace(",", ""))
    return float(match.group()) if match else np.nan


def parse_rating(text):
    # 'Three' -> 3 ; anything else -> NaN
    if not isinstance(text, str):
        return np.nan
    return RATING_MAP.get(text.strip().lower(), np.nan)


def parse_in_stock(text):
    # 'In stock' -> True, 'Out of stock' -> False, anything else -> None (unknown)
    if not isinstance(text, str):
        return None
    t = text.strip().lower()
    if t.startswith("out of stock"):
        return False
    if t.startswith("in stock"):
        return True
    return None


def clean_books(raw):
    df = raw.copy()
    df["title"] = df["title"].astype("string").str.strip()
    df["price_gbp"] = df["price"].map(parse_price).astype(float)
    df["rating"] = df["star_rating"].map(parse_rating).astype(float)
    df["in_stock"] = df["availability"].map(parse_in_stock)

    report = {"rows_in": len(df)}

    # --- drop rows that cannot be identified / linked ---
    before = len(df)
    df = df.dropna(subset=["title", "category"])
    df = df[df["title"].str.len() > 0]
    report["dropped_missing_title_or_category"] = before - len(df)

    # --- boolean field: unknown availability -> drop the row ---
    before = len(df)
    df = df[df["in_stock"].notna()]
    report["dropped_unparseable_availability"] = before - len(df)

    # --- numeric fields: median imputation (category median, then overall median) ---
    df["imputed"] = df["price_gbp"].isna() | df["rating"].isna()
    report["price_imputed"] = int(df["price_gbp"].isna().sum())
    report["rating_imputed"] = int(df["rating"].isna().sum())
    for col in ["price_gbp", "rating"]:
        cat_median = df.groupby("category")[col].transform("median")
        df[col] = df[col].fillna(cat_median).fillna(df[col].median())
    df["rating"] = df["rating"].round().clip(1, 5)

    # --- duplicates ---
    before = len(df)
    df = df.drop_duplicates(subset=["title", "category"])
    report["dropped_duplicates"] = before - len(df)

    # --- final types ---
    df["price_gbp"] = df["price_gbp"].astype(float).round(2)
    df["rating"] = df["rating"].astype(int)
    df["in_stock"] = df["in_stock"].astype(bool)
    df["title"] = df["title"].astype(str)
    df["category"] = df["category"].astype(str)
    report["rows_out"] = len(df)
    cols = ["title", "category", "price_gbp", "rating", "in_stock", "imputed"]
    return df[cols].reset_index(drop=True), report

### Robustness check on deliberately messy rows

Before cleaning the real data, feed the cleaner a few made-up broken rows to prove it does not crash. (These test rows are **not** added to the dataset.)

In [5]:
messy = pd.DataFrame([
    {"title": "Good row",          "price": "£20.00",    "star_rating": "Four", "availability": "In stock",     "category": "Test"},
    {"title": "Bad price",         "price": "Price N/A", "star_rating": "Two",  "availability": "In stock",     "category": "Test"},
    {"title": "Bad rating",        "price": "£40.00",    "star_rating": "Zero", "availability": "Out of stock", "category": "Test"},
    {"title": "Bad availability",  "price": "£30.00",    "star_rating": "Five", "availability": "???",          "category": "Test"},
    {"title": None,                "price": "£10.00",    "star_rating": "One",  "availability": "In stock",     "category": "Test"},
])
test_clean, test_report = clean_books(messy)
print(test_report)
test_clean

{'rows_in': 5, 'dropped_missing_title_or_category': 1, 'dropped_unparseable_availability': 1, 'price_imputed': 1, 'rating_imputed': 1, 'dropped_duplicates': 0, 'rows_out': 3}


,title,category,price_gbp,rating,in_stock,imputed
0,Good row,Test,20.0,4,True,False
1,Bad price,Test,30.0,2,True,True
2,Bad rating,Test,40.0,3,False,True


Result: the bad price and bad rating were filled with the category median (and flagged `imputed=True`), while the unknown-availability row and the untitled row were dropped — no crash.

In [6]:
books, clean_report = clean_books(raw_df)
print("Cleaning report:", clean_report)
print()
print(books.dtypes)
books.head()

Cleaning report: {'rows_in': 125, 'dropped_missing_title_or_category': 0, 'dropped_unparseable_availability': 0, 'price_imputed': 0, 'rating_imputed': 0, 'dropped_duplicates': 1, 'rows_out': 124}

title            str
category         str
price_gbp    float64
rating         int64
in_stock        bool
imputed         bool
dtype: object


,title,category,price_gbp,rating,in_stock,imputed
0,Sharp Objects,Mystery,47.82,4,True,False
1,"In a Dark, Dark Wood",Mystery,19.63,1,True,False
2,The Past Never Ends,Mystery,56.50,4,True,False
3,A Murder in Time,Mystery,16.64,1,True,False
4,The Murder of Roger Ackroyd (Hercule Poir...,Mystery,44.10,4,True,False


## 3. Convert GBP → INR with the fixed project rate

`price_inr = price_gbp × 105.50`. The rate **1 GBP = 105.50 INR** is an artificial constant defined by the assignment — not a live or historical market rate — so no API call or date is involved.

In [7]:
books["price_inr"] = (books["price_gbp"] * GBP_TO_INR).round(2)

# Quick sanity check: every row obeys the formula
assert np.allclose(books["price_inr"], books["price_gbp"] * GBP_TO_INR, atol=0.01)
assert books["rating"].between(1, 5).all()

print(books.dtypes)
print(f"\nFinal clean dataset: {len(books)} books across {books['category'].nunique()} categories")
books[["title", "category", "price_gbp", "price_inr", "rating", "in_stock"]].head(10)

title            str
category         str
price_gbp    float64
rating         int64
in_stock        bool
imputed         bool
price_inr    float64
dtype: object

Final clean dataset: 124 books across 4 categories


,title,category,price_gbp,price_inr,rating,in_stock
0,Sharp Objects,Mystery,47.82,5045.01,4,True
1,"In a Dark, Dark Wood",Mystery,19.63,2070.96,1,True
2,The Past Never Ends,Mystery,56.50,5960.75,4,True
3,A Murder in Time,Mystery,16.64,1755.52,1,True
4,The Murder of Roger Ackroyd (Hercule Poir...,Mystery,44.10,4652.55,4,True
5,The Last Mile (Amos Decker #2),Mystery,54.21,5719.16,2,True
6,That Darkness (Gardiner and Renner #1),Mystery,13.92,1468.56,1,True
7,Tastes Like Fear (DI Marnie Rome #3),Mystery,10.69,1127.79,1,True
8,A Time of Torment (Charlie Parker #14),Mystery,48.35,5100.92,5,True
9,A Study in Scarlet (Sherlock Holmes #1),Mystery,16.73,1765.02,2,True


## 4. Store in a normalized SQLite database

Two tables linked by a primary key / foreign key:

```
categories                         books
-----------                        -----
category_id  INTEGER PK  <─────┐   book_id     INTEGER PK
category_name TEXT UNIQUE      └── category_id INTEGER FK → categories.category_id
                                   title, price_gbp, price_inr, rating, in_stock
```

**Why normalize?** The category name is stored once in `categories`; each book only stores a small integer pointing to it. Renaming a category means changing one row, and a typo cannot create a “new” category by accident.

The database is rebuilt from scratch every time this notebook runs (`DROP TABLE IF EXISTS`), so this notebook is also the exact recreation script for `data/books.db`.

In [8]:
SCHEMA = '''
PRAGMA foreign_keys = ON;
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS categories;

CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY,
    category_name TEXT NOT NULL UNIQUE
);

CREATE TABLE books (
    book_id     INTEGER PRIMARY KEY,
    title       TEXT    NOT NULL,
    price_gbp   REAL    NOT NULL,
    price_inr   REAL    NOT NULL,
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    in_stock    INTEGER NOT NULL CHECK (in_stock IN (0, 1)),
    category_id INTEGER NOT NULL REFERENCES categories(category_id)
);
'''

# In-memory versions of the two tables (also reused later for the pd.merge comparison)
categories_df = (pd.DataFrame({"category_name": sorted(books["category"].unique())})
                   .rename_axis("category_id").reset_index())
categories_df["category_id"] += 1                      # ids start at 1

books_df = books.merge(categories_df, left_on="category", right_on="category_name")
books_df = books_df.sort_values(["category_id", "title"]).reset_index(drop=True)
books_df.insert(0, "book_id", books_df.index + 1)
books_df = books_df[["book_id", "title", "price_gbp", "price_inr", "rating", "in_stock", "category_id"]]
books_df["in_stock"] = books_df["in_stock"].astype(int)   # SQLite has no bool type: 1 = True, 0 = False

with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(SCHEMA)
    conn.executemany("INSERT INTO categories (category_id, category_name) VALUES (?, ?)",
                     categories_df.itertuples(index=False, name=None))
    conn.executemany("INSERT INTO books VALUES (?, ?, ?, ?, ?, ?, ?)",
                     books_df.itertuples(index=False, name=None))
    conn.commit()

    n_cat = conn.execute("SELECT COUNT(*) FROM categories").fetchone()[0]
    n_books = conn.execute("SELECT COUNT(*) FROM books").fetchone()[0]
    fk_problems = conn.execute("PRAGMA foreign_key_check").fetchall()

print(f"Saved {DB_PATH}: {n_cat} categories, {n_books} books, foreign-key problems: {len(fk_problems)}")
categories_df

Saved data\books.db: 4 categories, 124 books, foreign-key problems: 0


,category_id,category_name
0,1,Fantasy
1,2,Historical Fiction
2,3,Mystery
3,4,Poetry


## 5. Query the database

Six queries that together cover every required clause:

| # | Question | Clauses shown |
|---|---|---|
| Q1 | In-stock books under £20 | `SELECT … WHERE` (with `AND`) |
| Q2 | The 10 most expensive books | `ORDER BY … DESC`, `LIMIT` |
| Q3 | Which star ratings exist | `DISTINCT` |
| Q4 | 4–5 star books priced £20–£30 | `BETWEEN`, `IN` |
| Q5 | Books rated 4+ with their category name | `JOIN` |
| Q6 | Summary per category | `JOIN`, `GROUP BY`, aggregates |

Every query string and its output are also written to `outputs/query_results.md`.

In [9]:
QUERIES = {
    "Q1 - In-stock books under £20 (SELECT / WHERE)": '''
SELECT title, price_gbp, rating
FROM books
WHERE in_stock = 1 AND price_gbp < 20
ORDER BY price_gbp;''',

    "Q2 - Top 10 most expensive books (ORDER BY / LIMIT)": '''
SELECT title, price_gbp, price_inr
FROM books
ORDER BY price_gbp DESC
LIMIT 10;''',

    "Q3 - Distinct star ratings in the catalogue (DISTINCT)": '''
SELECT DISTINCT rating
FROM books
ORDER BY rating;''',

    "Q4 - 4 or 5 star books priced between £20 and £30 (BETWEEN / IN)": '''
SELECT title, price_gbp, rating
FROM books
WHERE price_gbp BETWEEN 20 AND 30
  AND rating IN (4, 5)
ORDER BY price_gbp DESC;''',

    "Q5 - Highest-rated books (4+ stars) with their category (JOIN)": '''
SELECT c.category_name, b.title, b.rating, b.price_gbp, b.price_inr
FROM books AS b
JOIN categories AS c ON b.category_id = c.category_id
WHERE b.rating >= 4
ORDER BY c.category_name, b.rating DESC, b.price_gbp DESC, b.title;''',

    "Q6 - Per-category summary (JOIN / GROUP BY)": '''
SELECT c.category_name,
       COUNT(*)                       AS n_books,
       ROUND(AVG(b.price_gbp), 2)     AS avg_price_gbp,
       ROUND(AVG(b.price_inr), 2)     AS avg_price_inr,
       ROUND(AVG(b.rating), 2)        AS avg_rating,
       ROUND(100.0 * AVG(b.in_stock), 1) AS pct_in_stock
FROM books AS b
JOIN categories AS c ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY n_books DESC;''',
}

results = {}
with sqlite3.connect(DB_PATH) as conn:
    # Q1 with a plain sqlite3 cursor, to show the low-level way too
    name1 = list(QUERIES)[0]
    cur = conn.execute(QUERIES[name1])
    cols = [d[0] for d in cur.description]
    results[name1] = pd.DataFrame(cur.fetchall(), columns=cols)

    # The rest are read straight into DataFrames with pd.read_sql
    for name, sql in list(QUERIES.items())[1:]:
        results[name] = pd.read_sql(sql, conn)

for name, df_out in results.items():
    print("=" * 100)
    print(name)
    print(QUERIES[name].strip())
    print("-" * 100)
    print(f"{len(df_out)} rows")
    print(df_out.head(15).to_string(index=False))
    print()

Q1 - In-stock books under £20 (SELECT / WHERE)
SELECT title, price_gbp, rating
FROM books
WHERE in_stock = 1 AND price_gbp < 20
ORDER BY price_gbp;
----------------------------------------------------------------------------------------------------
23 rows
                                                                   title  price_gbp  rating
                                    Tastes Like Fear (DI Marnie Rome #3)      10.69       1
                                              Hide Away (Eve Duncan #20)      11.84       1
                        Every Heart a Doorway (Every Heart A Doorway #1)      12.16       5
                                                       The Girl You Lost      12.29       5
                                         Sister Sable (The Mad Queen #1)      13.33       3
                        Princess Between Worlds (Wide-Awake Princess #5)      13.34       5
                                                       Playing with Fire      13.71       3
       

In [10]:
# Save every query string + its full output to a Markdown file
lines = ["# SQL query results\n",
         f"Database: `data/books.db` — {n_books} books, {n_cat} categories. "
         "Generated by `books_pipeline.ipynb`.\n"]
for name, df_out in results.items():
    lines.append(f"\n## {name}\n")
    lines.append("```sql\n" + QUERIES[name].strip() + "\n```\n")
    lines.append(f"**{len(df_out)} rows**\n")
    lines.append(df_out.to_markdown(index=False) + "\n")
(OUT_DIR / "query_results.md").write_text("\n".join(lines), encoding="utf-8")
print("Wrote", OUT_DIR / "query_results.md")

Wrote outputs\query_results.md


## 6. Same JOIN two ways: `pd.read_sql` vs `pd.merge`

Q5 was answered by SQL. Here the same answer is rebuilt **without any SQL**, using `pd.merge` on the in-memory `books_df` and `categories_df` DataFrames, then the two results are compared cell by cell.

In [11]:
with sqlite3.connect(DB_PATH) as conn:
    join_sql = pd.read_sql(QUERIES["Q5 - Highest-rated books (4+ stars) with their category (JOIN)"], conn)

join_pandas = (
    books_df.merge(categories_df, on="category_id", how="inner")      # the JOIN
            .query("rating >= 4")                                      # the WHERE
            .sort_values(["category_name", "rating", "price_gbp", "title"],
                         ascending=[True, False, False, True])         # the ORDER BY
            [["category_name", "title", "rating", "price_gbp", "price_inr"]]
            .reset_index(drop=True)
)

# Side by side (first 10 rows)
side_by_side = pd.concat({"pd.read_sql (SQL JOIN)": join_sql, "pd.merge (no SQL)": join_pandas}, axis=1)
display(side_by_side.head(10))

# Cell-by-cell comparison. assert_frame_equal raises an error if anything differs.
pd.testing.assert_frame_equal(join_sql, join_pandas, check_dtype=False)
print(f"MATCH: both approaches return the same {len(join_sql)} rows x {join_sql.shape[1]} columns")

pd.read_sql (SQL JOIN)                                                                          pd.merge (no SQL)  \
           category_name                                         title rating price_gbp price_inr     category_name   
0                Fantasy  The False Prince (The Ascendance Trilogy #1)      5     56.00   5908.00           Fantasy   
1                Fantasy         Paper and Fire (The Great Library #2)      5     49.45   5216.98           Fantasy   
2                Fantasy  Harry Potter and the Half-Blood Prince (H...      5     48.75   5143.12           Fantasy   
3                Fantasy      The Beast (Black Dagger Brotherhood #14)      5     46.08   4861.44           Fantasy   
4                Fantasy                        The Star-Touched Queen      5     46.02   4855.11           Fantasy   
5                Fantasy      King's Folly (The Kinsman Chronicles #1)      5     39.61   4178.85           Fantasy   
6                Fantasy  Demigods & Magicians: Percy and Annabeth ...      5     37.51   3957.30           Fantasy   
7                Fantasy  Princess Between Worlds (Wide-Awake Princ...      5     13.34   1407.37           Fantasy   
8                Fantasy  Every Heart a Doorway (Every Heart A Door...      5     12.16   1282.88           Fantasy   
9                Fantasy                           Myriad (Prentor #1)      4     58.75   6198.12           Fantasy   

                                                                            
                                          title rating price_gbp price_inr  
0  The False Prince (The Ascendance Trilogy #1)      5     56.00   5908.00  
1         Paper and Fire (The Great Library #2)      5     49.45   5216.98  
2  Harry Potter and the Half-Blood Prince (H...      5     48.75   5143.12  
3      The Beast (Black Dagger Brotherhood #14)      5     46.08   4861.44  
4                        The Star-Touched Queen      5     46.02   4855.11  
5      King's Folly (The Kinsman Chronicles #1)      5     39.61   4178.85  
6  Demigods & Magicians: Percy and Annabeth ...      5     37.51   3957.30  
7  Princess Between Worlds (Wide-Awake Princ...      5     13.34   1407.37  
8  Every Heart a Doorway (Every Heart A Door...      5     12.16   1282.88  
9                           Myriad (Prentor #1)      4     58.75   6198.12

MATCH: both approaches return the same 56 rows x 5 columns


## 7. Summary

* Scraped every book in 4 categories with `requests` + `BeautifulSoup` (following pagination), saved raw to `data/raw_books.csv`.
* Cleaned into typed columns `price_gbp` (float), `rating` (int 1–5), `in_stock` (bool); bad numeric values → category-median imputation, unknown availability → row dropped.
* Added `price_inr` using the fixed project rate **1 GBP = 105.50 INR**.
* Loaded a normalized two-table SQLite schema (`categories` PK ⟵ `books.category_id` FK) into `data/books.db`.
* Ran 6 SQL queries covering WHERE, ORDER BY, LIMIT, DISTINCT, BETWEEN, IN, JOIN and GROUP BY; outputs saved to `outputs/query_results.md`.
* Showed that the JOIN result from `pd.read_sql` and from `pd.merge` are identical.